## Section 1 - Setup and Data Loading

In [ ]:
from __future__ import annotations

import importlib.util
import sys
from pathlib import Path

import pandas as pd

EXPERIMENTS_DIR = Path("__file__").resolve().parent if "__file__" in dir() else Path.cwd()

EXPERIMENT_DIRS = {
    "kb_derivation_performance": EXPERIMENTS_DIR / "kb_derivation_performance",
    "consistency_answer_section": EXPERIMENTS_DIR / "consistency_answer_section",
}

def _add_to_path(d: Path) -> None:
    s = str(d)
    if s not in sys.path:
        sys.path.insert(0, s)

def _load_metrics_module(name: str, filepath: Path):
    spec = importlib.util.spec_from_file_location(name, filepath)
    mod = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(mod)
    return mod

print("Experiments directory:", EXPERIMENTS_DIR)

## Load base CSV (Prolog programs)

In [ ]:
BASE_CSV = EXPERIMENTS_DIR / "results_prolog-computed.csv"
df_programs = pd.read_csv(BASE_CSV)

print(f"Base CSV: {BASE_CSV}")
print(f"Shape: {df_programs.shape}")
print(f"Columns: {list(df_programs.columns)}")
print(f"Unique question_index: {df_programs['question_index'].nunique()}")

df_programs[["question_index", "model"]].head(5)

## Section 2 - KB Derivation Performance

Deriving the conclusions.

In [ ]:
_add_to_path(EXPERIMENT_DIRS['kb_derivation_performance'])

from kb_derivation_performance.main import get_results

# ! commenting the line below to avoid re-computing the data through an LLM
df_der_results = get_results(save=False)
df_der_results.head()

Computing the scores against the ground truth

In [ ]:
from kb_derivation_performance._analysis.assess_kb_response_prompt import build_assess_kb_response_prompt
import pandas as pd
from langchain_openai import ChatOpenAI
import os
from dotenv import load_dotenv
from tqdm import tqdm

load_dotenv(".env")

MODEL_NAME = "openai/gpt-4o"
TEMPERATURE = 0
OPENROUTER_BASE_URL = "https://openrouter.ai/api/v1"
OPENROUTER_API_KEY = os.getenv("OPENROUTER_API_KEY")

df_der_results["label"] = None  # create new column first

# ! commenting the code below to avoid re-computing the results through LLM
for idx, row in tqdm(df_der_results.iterrows(), total=len(df_der_results)):
    question_index = row['question_index']
    kb_conclusions = row['kb_conclusions']
    target_nl_conclusion = row['target_nl_conclusion']

    # print("--------------------------")
    # print(f"# {question_index}, {kb_conclusions}, {target_nl_conclusion}")

    # Case 1: Empty KB response
    if kb_conclusions == "[]" or len(kb_conclusions) == 0:  # empty list
        df_der_results.at[idx, "label"] = "Inference Error"
        continue

    # Take first KB conclusion
    kb_response = kb_conclusions[0]

    system_prompt, user_prompt = build_assess_kb_response_prompt(
        kb_response, target_nl_conclusion
    )

    messages = [
        ("system", system_prompt),
        ("user", user_prompt)
    ]

    llm = ChatOpenAI(
        model=MODEL_NAME,
        temperature=TEMPERATURE,
        api_key=OPENROUTER_API_KEY,
        base_url=OPENROUTER_BASE_URL
    )

    result = llm.invoke(messages)

    llm_response = result.content.strip()

    if llm_response == "True":
        df_der_results.at[idx, "label"] = "True"
    else:
        df_der_results.at[idx, "label"] = "False"

# ! uncomment these lines to load the pre-computed DF
# df_der_results = pd.read_csv(os.path.join(
#     "kb_derivation_performance", 
#     "_analysis",
#     "conclusion_llm_verification_23_02_2026.csv"
# ))

In [ ]:
df_der_results['label'].value_counts()

## Section 3 - LRM Output Consistency

In [ ]:
_add_to_path(EXPERIMENT_DIRS["consistency_answer_section"])
from consistency_answer_section.consistency_analyzer import ConsistencyAnalyzer
_con_mod = _load_metrics_module("con_metrics", EXPERIMENT_DIRS["consistency_answer_section"] / "metrics.py")
con_score_fn = _con_mod.consistency_score

print("consistency analyzer imported successfully.")

In [ ]:
judge_llm = ChatOpenAI(
    model="openai/gpt-4o",
    temperature=0,
    base_url="https://openrouter.ai/api/v1",
    api_key=OPENROUTER_API_KEY
)

In [ ]:
df_con = df_programs.copy()

In [ ]:
from tqdm import tqdm
from consistency_answer_section.metrics import consistency_score

real_scores = []
i = 1
for _, row in tqdm(df_con.iterrows(), total=len(df_con)):
    print("\n ------------------- \n")
    result = ConsistencyAnalyzer(
        reasoning=row["reasoning"],
        answer=row["answer"],
        llm=judge_llm,
    ).analyze()

    con_score = consistency_score(result)

    print(f"LRM answer={row['answer']}")
    print(f"con_score={con_score}")

    real_scores.append(con_score)

df_con["consistency_score"] = real_scores

print(f"Evaluable rows : {df_con['consistency_score'].notna().sum()} / {len(df_con)}")
print(f"Mean score     : {df_con['consistency_score'].mean():.4f}")
df_con[["question_index", "consistency_score"]].head(10)

In [ ]:
df_con['consistency_score'].describe().round(3)

In [ ]:
df_con['consistency_score'].fillna(0).describe().round(3)